# PEKA reproduction — Kaggle T4 x2

Repo: `duyh80456-code/peka-rebuild` (public, track A / breast only)

**Muc tieu:** baseline H-optimus-0 **~0.624**  ->  PEKA **~0.698**

## Cach dung
Doi DUY NHAT `RUN_PHASE` o cell duoi, roi **Run All**.
Xong thi **Save Version**, va attach Output do lam **Input** cho session sau.

| PHASE | Lam gi | Thoi gian | Can GPU? |
|---|---|---|---|
| 1 | download HEST + cat patch + align gene | 4-6h | **khong** |
| 2 | scFoundation embedding | 3-5h | co |
| 3 | kmeans + smoke + train — **lap lai den khi xong** | 12h/session | co |
| 4 | extract feature + HVG + eval -> **PCC** | ~1h | co |
| 5 | baseline image_encoder (doi chung) | ~1h | co |

> PHASE 1 khong dung GPU. Chay no o session **Accelerator: None** de tiet kiem
> quota GPU (~30h/tuan) cho cac phase sau.

## Truoc khi chay
1. Bat **Internet** (va **GPU T4 x2** tu PHASE 2 tro di)
2. Tao Kaggle Secret **`HF_TOKEN`**
3. Vao HuggingFace **chap nhan dieu khoan** (khong lam se loi 403 giua chung):
   - huggingface.co/datasets/MahmoodLab/hest
   - huggingface.co/bioptimus/H-optimus-0

## Train nhieu session
PHASE 3 tu tim `last.ckpt` va truyen `--resume`. Cu chay lai PHASE 3
cho den khi log bao du `EPOCHS`. State optimizer/scheduler/epoch duoc khoi phuc day du.


In [ ]:
# ================= CHI DOI O DAY =================
RUN_PHASE  = 1        # 1..5
EPOCHS     = 50       # paper: 50
BATCH_SIZE = 8        # giam ve 4 neu OOM
ACCUM      = 4        # batch hieu dung = BATCH_SIZE * ACCUM = 32 (paper)
PEFT       = "bone"   # phuong phap cua paper (Block-Affine)
ENCODER    = "H-optimus-0"
# =================================================
print("RUN_PHASE =", RUN_PHASE)

In [ ]:
import os, subprocess, sys, glob
from pathlib import Path

REPO_URL = "https://github.com/duyh80456-code/peka-rebuild.git"
REPO     = Path("/kaggle/working/peka-rebuild")
SCRATCH  = Path("/kaggle/tmp/peka")        # KHONG duoc luu -> de HEST tho o day
OUT      = Path("/kaggle/working/OUTPUT")  # duoc luu thanh Output
EXP      = ENCODER + "_" + PEFT + "_breast_in_hest_joint"
SCRATCH.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, env=None, cwd=REPO):
    print("+", " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip()
assert os.environ["HF_TOKEN"], "Thieu Kaggle Secret HF_TOKEN"

if not (REPO / ".git").is_dir():
    run(["git", "clone", "--recursive", REPO_URL, str(REPO)], cwd="/kaggle/working")
else:
    run(["git", "pull", "--ff-only"])

# QUAN TRONG: --no-deps. Kaggle da co san torch/lightning/scanpy/hydra_zen...
# Neu de pip giai deps cua peka (numpy, pandas... deu khong ghim) no se ghi de
# len moi truong Kaggle va lam VO numpy (loi "_center from numpy._core.umath").
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

# Chi cai dung nhung gi Kaggle THUC SU thieu (pip da liet ke ro khi dung
# --no-deps). Tat ca deu la goi python thuan, khong dong vao numpy/scipy:
#   peft <0.19       ban moi hon da bo BoneConfig, ma "bone" la pp cua paper
#   hydra-zen        tang config cua peka
#   biomart          dung khi align ten gene
#   local_attention  scFoundation can
#   hestcore         HEST nap qua sys.path nen deps cua no khong duoc cai
run([sys.executable, "-m", "pip", "install", "-q",
     "peft>=0.17,<0.19", "hydra-zen", "biomart", "local_attention",
     "hestcore==1.0.3", "hf_transfer",
     "openslide-python", "mygene", "loguru", "einops-exts"])

# Kiem trong tien trinh RIENG (giong luc chay script that), khong import vao
# kernel cua notebook -> tranh dinh module da nap tu truoc.
run([sys.executable, "-c",
     "import numpy, peft, hydra_zen, biomart, local_attention; "
     "print('numpy', numpy.__version__, '| peft', peft.__version__); "
     "assert hasattr(peft, 'BoneConfig'), 'peft thieu BoneConfig'"])

(REPO / ".env").write_text(
    "HF_TOKEN=" + os.environ["HF_TOKEN"] + "\n"
    "HEST1K_STORAGE_PATH=" + str(SCRATCH) + "/HEST1K\n"
    "WANDB_API_KEY=\nWANDB_ENTITY=\n")

# --- CHAN DOAN: /kaggle/input dang co gi ------------------------------------
print("=== /kaggle/input ===")
_inp = Path("/kaggle/input")
_kids = sorted(_inp.glob("*")) if _inp.is_dir() else []
if not _kids:
    print("   RONG! Chua attach input nao.")
    print("   -> Cot phai > + Add Input > Your Work > chon notebook nay > bam +")
for _a in _kids:
    print("  ", _a.name + "/")
    for _b in sorted(_a.glob("*"))[:8]:
        print("      ", _b.name + ("/" if _b.is_dir() else ""))
        if _b.is_dir():
            for _c in sorted(_b.glob("*"))[:6]:
                print("          ", _c.name + ("/" if _c.is_dir() else ""))
print("=====================")

# --- DATA/Pretrained SONG O /kaggle/tmp, KHONG o /kaggle/working ------------
# /kaggle/working chi co quota ~20 GiB. DATA (~7 GiB) + checkpoint (~5 GiB moi
# cai, Lightning giu 2 cai: best + last) vuot quota -> chet giua chung voi
# "OSError: [Errno 28] No space left on device". /kaggle/tmp dung chung o dia
# ~70 GiB va KHONG bi quota, nen de o do roi symlink vao repo.
def stage(kind):
    real = SCRATCH / kind
    real.mkdir(parents=True, exist_ok=True)
    link = REPO / kind
    if link.is_symlink():
        link.unlink()
    elif link.is_dir():
        subprocess.run(["rsync", "-a", str(link) + "/", str(real) + "/"], check=False)
        subprocess.run(["rm", "-rf", str(link)], check=False)
    link.symlink_to(real)
    return real

def restore(kind, probe):
    real = stage(kind)
    pats = ["/".join(["*"] * d) + "/" + kind for d in range(1, 7)]
    for pat in pats:
        for cand in sorted(Path("/kaggle/input").glob(pat)):
            if not cand.is_dir():
                continue
            if probe and not (cand / probe).exists():
                continue
            print("Khoi phuc", kind, "tu", cand)
            subprocess.run(["rsync", "-a", "--ignore-existing", "--exclude", "HEST1K",
                            str(cand) + "/", str(real) + "/"], check=False)
            return True
    return False

RESTORED_DATA = restore("DATA", "breast/breast_in_hest/aligned_adata")
restore("Pretrained", EXP)

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src")
env["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# BAT BUOC dat bien moi truong THAT, khong duoc chi dua vao .env:
# peka/__init__.py nap .env SAU khi paths.py da tinh xong HEST1K_STORAGE_PATH,
# nen gia tri trong .env bi bo qua -> script 10 va 11 tro ve hai cho khac nhau.
env["HEST1K_STORAGE_PATH"] = str(SCRATCH / "HEST1K")
# Ep 1 GPU: repo khong cau hinh devices/strategy, Lightning se TU BAT DDP khi
# thay 2 GPU. Dataset dung h5py handle + fork -> DDP de deadlock.
GPU0 = dict(env)
GPU0["CUDA_VISIBLE_DEVICES"] = "0"

DATA       = REPO / "DATA"
PRETRAINED = REPO / "Pretrained"
LAST       = PRETRAINED / EXP / "last.ckpt"
EMB_DIR    = DATA / "breast/breast_in_hest/peka_embed/H0/scFoundation/default_model"
print("Repo san sang. Checkpoint se o:", LAST)

# --- TON KHO: kiem chinh xac cai gi da khoi phuc duoc ------------------------
# Da chay nhieu version thi dung doan; de no dem va bao thang.
B = DATA / "breast/breast_in_hest"
M = B / "scLLM_embed/scFoundation/default_model"
INV = {
    "patches(.h5)":     len(list(B.glob("patches/*.h5"))),
    "aligned_adata":    len(list(B.glob("aligned_adata/*.h5ad"))),
    "paired_seq":       len(list(M.glob("paired_seq/*.h5ad"))),
    "embeddings(.npy)": len(list(M.glob("embeddings/*.npy"))),
    "peka_embed":       len(list((B / "peka_embed").rglob("*.npy"))),
    "last.ckpt":        len(list(PRETRAINED.rglob("last.ckpt"))),
}
print("--- TON KHO sau khoi phuc (moc chuan: 8 slide) ---")
for k, v in INV.items():
    print("   %-18s %d" % (k, v))
print("--------------------------------------------------")

In [ ]:
try:
    if RUN_PHASE == 1 and INV["aligned_adata"] == 8 and INV["patches(.h5)"] == 8:
        # Da co du du lieu tu dataset dinh kem -> khong lam lai.
        # (buoc 11 goi web service Ensembl BioMart, hay hong that thuong)
        print(">>> PHASE 1 DA XONG TU TRUOC (8 patches + 8 aligned_adata).")
        print(">>> Doi RUN_PHASE = 2 roi chay lai.")

    elif RUN_PHASE == 1:
        run([sys.executable, "scripts/00_check_env.py"], env=env)
        run([sys.executable, "scripts/10_download_hest1k.py",
             "--minimal", "--max-workers", "16"], env=env)
        run([sys.executable, "scripts/11_build_breast_dataset.py"], env=env)

        # --- KIEM 2 GIA DINH truoc khi dot GPU -------------------------------
        # Chay trong TIEN TRINH RIENG: kernel notebook nap numpy tu luc khoi
        # dong, sau do pip thay file ben duoi -> import anndata trong kernel vo.
        check = REPO / "_check_f1_f23.py"
        check.write_text(chr(10).join([
            'import glob, sys',
            'import h5py, anndata',
            "root = sys.argv[1] + '/breast/breast_in_hest'",
            "ph = sorted(glob.glob(root + '/patches/*.h5'))[0]",
            'f = h5py.File(ph)',
            "print('[F1] patch dtype =', f['img'].dtype, ' max =', f['img'][0].max())",
            "print('     uint8/255 => anh KHONG duoc chuan hoa truoc khi vao backbone')",
            "img_bc = [b.decode() if isinstance(b, bytes) else str(b) for b in f['barcode'][:, 0]]",
            'f.close()',
            "ad = sorted(glob.glob(root + '/aligned_adata/*.h5ad'))[0]",
            'seq = anndata.read_h5ad(ad).obs_names.to_numpy()',
            'keep = set(img_bc) & set(seq)',
            'a = [b for b in img_bc if b in keep][:200]',
            'b = [b for b in seq if b in keep][:200]',
            "print('[F23] thu tu barcode  patch == adata ?', a == b)",
            'if a != b:',
            "    raise SystemExit('[F23] THU TU BARCODE LECH. Feature se ghep NHAM nhan cua spot khac -> PCC thap gia tao. DUNG LAI, dieu tra truoc khi train.')",
            "print('     OK - feature va nhan se ghep dung.')",
        ]))
        run([sys.executable, str(check), str(DATA)], env=env)

    elif RUN_PHASE == 2:
        assert INV["aligned_adata"] == 8, (
            "Can 8 aligned_adata, chi thay %d. Version dang attach KHONG phai ban "
            "PHASE 1 hoan chinh -> doi sang version co log 'PHASE 1 xong'."
            % INV["aligned_adata"])
        run([sys.executable, "scripts/12_compute_scfoundation_emb.py"], env=GPU0)

    elif RUN_PHASE == 3:
        assert INV["embeddings(.npy)"] == 8 and INV["paired_seq"] == 8, (
            "Can 8 embedding + 8 paired_seq, chi thay %d + %d. Chay PHASE 2 truoc, "
            "hoac attach dung version da hoan thanh PHASE 2."
            % (INV["embeddings(.npy)"], INV["paired_seq"]))
        run([sys.executable, "scripts/13_kmeans_cluster_labels.py", "--use_gpu"], env=GPU0)
        run([sys.executable, "scripts/14_smoke_test_loader.py",
             "--num_batches", "2", "--num_workers", "0"], env=env)

        cmd = [sys.executable, "scripts/30_train_phase2_kd.py",
               "--encoder", ENCODER, "--peft", PEFT,
               "--epochs", str(EPOCHS),
               "--batch_size", str(BATCH_SIZE),
               "--accumulate_grad_batches", str(ACCUM),
               "--num_workers", "2",
               "--with_logger", "csv"]
        if LAST.is_file():
            print(">>> RESUME tu", LAST)
            cmd += ["--resume", str(LAST)]
        else:
            print(">>> Bat dau train tu dau")
        run(cmd, env=GPU0)

    elif RUN_PHASE == 4:
        assert LAST.is_file(), ("Khong thay checkpoint " + str(LAST) +
                                " -> attach version da chay xong PHASE 3.")
        # BAY: script 50 (--probe_only) tim feature trong peka_embed/H0/, nhung
        # script 40 mac dinh ghi vao peka_embed/<encoder>_<peft>/ -> ep --output_dir.
        run([sys.executable, "scripts/40_extract_peka_features.py",
             "--encoder", ENCODER, "--peft", PEFT, "--ckpt", str(LAST),
             "--output_dir", str(EMB_DIR)], env=GPU0)
        run([sys.executable, "scripts/41_compute_hvg_top50.py"], env=env)
        run([sys.executable, "scripts/50_eval_gene_regression_kfold.py",
             "--encoder", ENCODER, "--peft", PEFT, "--probe_only"], env=env)
        print("\n>>> So sanh voi paper: PEKA breast = 0.698")

    elif RUN_PHASE == 5:
        # Feature baseline duoc trich boi Exp_helper/5_patch_feature_embeder.py --
        # script legacy tinh path tu cwd, doc lai truoc khi chay.
        run([sys.executable, "scripts/50_eval_gene_regression_kfold.py",
             "--encoder", ENCODER, "--feature_type", "image_encoder",
             "--probe_only"], env=env)
        print("\n>>> So sanh voi paper: H-optimus-0 dong bang breast = 0.624")
finally:
    # /kaggle/working co quota ~20 GiB -> CHI mang sang session sau nhung thu
    # khong tao lai duoc. DATA da dong bang trong input thi khong chep lai.
    def sync(src, dst, extra=()):
        src, dst = Path(src), Path(dst)
        if not src.exists():
            return
        dst.parent.mkdir(parents=True, exist_ok=True)
        tail = "/" if src.is_dir() else ""
        subprocess.run(["rsync", "-a"] + list(extra) +
                       [str(src) + tail, str(dst) + tail], check=False)

    sync(LAST, OUT / "Pretrained" / EXP / "last.ckpt", ["-L"])          # ~5 GiB, de resume
    sync(REPO / "OUTPUT", OUT / "RUNS", ["--ignore-existing"])  # lora + csv log, nho
    if not RESTORED_DATA:
        sync(SCRATCH / "DATA", OUT / "DATA",
             ["--ignore-existing", "--exclude", "HEST1K"])      # phase 1/2: dong bang DATA
    else:
        sync(B / "peka_embed",                                  # phase 4: chi feature
             OUT / "DATA/breast/breast_in_hest/peka_embed", ["--ignore-existing"])

    def gib(p):
        p = Path(p)
        if not p.is_dir():
            return 0.0
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1024**3
    print("OUTPUT (co quota) : %.2f GiB / ~20 GiB" % gib(OUT))
    print("/kaggle/tmp       : %.2f GiB (khong tinh quota)" % gib(SCRATCH))
    print("PHASE", RUN_PHASE, "-> Save Version, roi attach Output nay cho session sau.")
